# Writing Effective Agent Instructions

*Notebook 03*

Instructions are the primary way you shape agent behavior.

Clear instructions make behavior more consistent and easier to debug.

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 📋 Part 1: Durable Instructions & Completion Criteria

Agent instructions guide every model turn.

Durable instructions work across many inputs.

They define the format, length, and completion criteria.

### Vague vs Durable

#### Run Vague Agent

In [ ]:
message = """Summarize this for me:
AI agents combine a language model with tools, memory, and instructions to accomplish tasks.
Unlike a plain chatbot, an agent can call functions, look up data, run code, and chain multiple steps
together to reach a goal. Modern agent frameworks handle the orchestration (deciding which tool to
call, passing results between steps, and recovering from errors) so developers can focus on defining
the agent's behavior rather than the control flow."""

agent_vague = Agent(
    name="VagueAgent",
    instructions="Be helpful and summarize things.",
    model=MODEL
)

vague_result = await Runner.run(agent_vague, input=message)

print("📌 Vague instructions:")
print(vague_result.final_output)

#### Run Durable Agent

In [ ]:
durable_instructions = (
    "You are a technical writing assistant.\n"
    "When asked to summarize, produce exactly one sentence.\n"
    "Use plain language: no jargon unless the user introduced it.\n"
    "Do not add commentary or follow-up questions."
)

agent_durable = Agent(
    name="DurableAgent",
    instructions=durable_instructions,
    model=MODEL
)

durable_result = await Runner.run(agent_durable, input=message)

print("📌 Durable instructions:")
print(durable_result.final_output)

### 💡 Key Takeaway

Completion criteria define the expected format, length, and stopping condition.

---

## ❓ Part 2: Ask Before Guessing

A good assistant does not invent missing details.

Tell it exactly which fields are required.

This demo has no calendar tool, so it only asks or confirms in text.

#### Run Agent with Incomplete Request

In [ ]:
clarify_instructions = (
    "You are a scheduling assistant.\n"
    "Before booking anything, confirm: the date, time, and attendees.\n"
    "If any of these are missing, ask for them. Do not assume.\n"
    "If all required details are present, confirm the booking details back to the user without asking for more information.\n"
    "When confirming, repeat the date back in the user's own words."
)

agent_clarify = Agent(
    name="SchedulingAgent",
    instructions=clarify_instructions,
    model=MODEL
)

clarify_result = await Runner.run(agent_clarify, input="Book a meeting with the team.")

print(clarify_result.final_output)

#### Run Agent with Complete Request

In [ ]:
complete_result = await Runner.run(
    agent_clarify,
    input="Book a meeting with Sarah and John tomorrow at 2pm."
)

print(complete_result.final_output)

### 💡 Key Takeaway

List the required fields explicitly.

In these runs the agent asked when a field was missing, and confirmed when all were present.

---

## 🚫 Part 3: When NOT to Use a Tool

In Lesson 02, you saw the model choose whether to call a tool.

Instructions can guide that choice.

Name when to call the tool and when to answer directly.

In [ ]:
@function_tool
def lookup_product_price(product_name: str) -> str:
    """Look up the current price of a product by name."""
    print(f"  🔧 [tool] lookup_product_price called for {product_name}")
    prices = {
        "widget": "$9.99",
        "gadget": "$24.99",
        "doohickey": "$4.99"
    }
    return prices.get(product_name.lower(), "Product not found.")


tool_constraint_instructions = (
    "You are a product support assistant.\n"
    "Use the lookup_product_price tool only when the user asks about price.\n"
    "For general questions, answer directly without calling any tool."
)

agent_constrained = Agent(
    name="ProductAgent",
    instructions=tool_constraint_instructions,
    model=MODEL,
    tools=[lookup_product_price]
)

result_general = await Runner.run(agent_constrained, input="What is a widget?")

print("📌 General question:")
print(result_general.final_output)

result_price = await Runner.run(agent_constrained, input="How much does a widget cost?")

print("\n📌 Price question:")
print(result_price.final_output)


### 💡 Key Takeaway

Name the condition for tool use in the instructions.

---

## 🛑 Part 4: Refusal and Escalation Patterns

For scoped support, define both boundaries and next steps.

**Refusal**: the agent declines an out-of-scope request.

**Escalation**: the agent directs the user to an appropriate person or channel.

Instructions guide this behavior.

They do not enforce it.

In [ ]:
refusal_instructions = (
    "You are a customer support agent for a software product.\n"
    "Known product fact: two-factor authentication is supported through authenticator apps.\n"
    "Only answer questions about the product: features, pricing, and troubleshooting.\n"
    "If the user asks about anything outside this scope, politely decline and suggest they contact the appropriate resource.\n"
    "If the user reports a billing dispute or account access issue, tell them to contact billing@acme.example. Do not attempt to resolve it yourself."
)

agent_refusal = Agent(
    name="SupportAgent",
    instructions=refusal_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ SupportAgent ready")

#### Test: In-Scope Question

In [ ]:
result_inscope = await Runner.run(agent_refusal, input="Does your product support two-factor authentication?")

print("📌 In-scope:")
print(result_inscope.final_output)

#### Test: Out-of-Scope Question

In [ ]:
result_outofscope = await Runner.run(agent_refusal, input="Can you help me write a cover letter?")

print("📌 Out-of-scope:")
print(result_outofscope.final_output)

#### Test: Escalation Trigger

In [ ]:
result_escalate = await Runner.run(agent_refusal, input="I was charged twice last month.")

print("📌 Escalation:")
print(result_escalate.final_output)

### 💡 Key Takeaway

Written rules make behavior more consistent than implied expectations.

---

## 💪 Practice Exercises

### Exercise 1: Strict Formatter

*Covers: durable instructions, explicit format and completion criteria*

Write instructions for a fixed format: one-sentence summary followed by exactly three bullet points.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Strict Formatter
# --------------------------------------------------------------
# Objective: Build an agent with instructions that specify a fixed output format.

# TODO 1: Write instructions that require:
#            - One sentence summary
#            - Exactly three bullet points
#            - No additional commentary

# TODO 2: Create an Agent with those instructions

# TODO 3: Run it with: "Tell me about large language models"
#            and verify the output matches the format

# --- Write your code below this line ---

### Exercise 2: Scoped Support Agent

*Covers: refusal and escalation, scoping agent behavior*

Build an agent that:

- answers cooking questions

- refuses unrelated requests

- escalates recipe disputes

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Scoped Support Agent
# --------------------------------------------------------------
# Objective: Write instructions that define scope and an escalation path.

# TODO 1: Write instructions that:
#            - Limit the agent to cooking-related questions only
#            - Define a specific refusal message for out-of-scope questions
#            - Escalate recipe disputes to: recipes@example.com

# TODO 2: Create an Agent with those instructions

# TODO 3: Test with three inputs:
#            - "How do I make pasta carbonara?"  (in-scope)
#            - "What's the weather today?"        (out-of-scope)
#            - "Your carbonara recipe is wrong." (escalation)

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Durable instructions make output more predictable:**

- Specify format, length, and tone explicitly

- Define completion criteria so the agent knows when to stop

- Name the condition for tool use: "use X tool only when Y"
<br>
<br>

**Explicit requirements reduce guessing:**

- Tell the agent which missing inputs require a question before proceeding

- Specify what to ask for, not just "ask if unclear"
<br>
<br>

**Support refusals should give the user a next step:**

- Define scope, refusal conditions, and escalation conditions

- Give the user a clear next step

- Instructions guide this behavior but do not enforce policy

---

## 📍 Next Step

**Notebook 04: Pydantic Basics**  

Learn how Pydantic defines typed models.

Then see why the course uses them for structured agent output.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-03-writing-agent-instructions)

---